# Maze Solver - Solution Validation Analysis

# Automatically discovers all trained models in models/NxN/ and runs
# validation analysis for each one. Generates datasets on the fly,
# evaluates exact match vs valid solution rate, and saves results.

In [ ]:
import os
import sys
import re
import json
import torch
import subprocess
import numpy as np
from datetime import datetime
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt

sys.path.append('./src')

from config import get_config
from model import ResNetGPT2PrefixModel
from tokenizer import SimpleTokenizer
from dataset import MazeDataset, collate_fn
from solution_validator import validate_solution, evaluate_with_validation, print_evaluation_results
from data_utils import load_maze_dataset, print_dataset_info

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = SimpleTokenizer()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print(f"Device: {device}")

## Discover Models

In [ ]:
# Scan models/ directory for all NxN checkpoints
model_dirs = sorted([
    d for d in os.listdir('models')
    if os.path.isdir(f'models/{d}') and re.match(r'\d+x\d+', d)
])

print(f"Found {len(model_dirs)} model(s):")
for d in model_dirs:
    pth = f'models/{d}/resnet_gpt2_prefix.pth'
    size = os.path.getsize(pth) / 1e6 if os.path.exists(pth) else 0
    print(f"  models/{d}/resnet_gpt2_prefix.pth  ({size:.1f} MB)")

## Helper Functions

In [ ]:
def load_maze_grid(image_path, grid_rows, grid_cols):
    """Convert maze image to binary grid (0=path, 1=wall)."""
    img_rgb = Image.open(image_path).convert('RGB')
    img_array_rgb = np.array(img_rgb)
    img_gray = Image.open(image_path).convert('L')
    img_array_gray = np.array(img_gray)

    height, width = img_array_gray.shape
    cell_height = height // grid_rows
    cell_width = width // grid_cols

    grid = np.zeros((grid_rows, grid_cols), dtype=int)

    for row in range(grid_rows):
        for col in range(grid_cols):
            center_y = row * cell_height + cell_height // 2
            center_x = col * cell_width + cell_width // 2
            r, g, b = img_array_rgb[center_y, center_x]

            is_green = (g > 100 and g > r and g > b)
            is_blue = (b > 100 and b > r and b > g)

            if is_green or is_blue:
                grid[row, col] = 0
            else:
                pixel_value = img_array_gray[center_y, center_x]
                grid[row, col] = 1 if pixel_value < 127 else 0

    return grid


def analyze_one_model(grid_size, checkpoint_path):
    """
    Generate dataset, load model, evaluate, and return results dict.
    """
    cfg = get_config(grid_size)
    variations = {4: 150, 5: 100, 7: 50}.get(grid_size, 50)

    # Generate the dataset for this grid size
    print(f"\n{'='*70}")
    print(f"  ANALYZING {grid_size}x{grid_size}")
    print(f"{'='*70}")

    subprocess.run(
        ['python', 'generate_dataset.py', '--size', str(grid_size),
         '--variations', str(variations), '--seed', '69'],
        check=True, capture_output=True
    )
    print(f"Generated {grid_size}x{grid_size} dataset")

    # Load test data
    test_entries, metadata = load_maze_dataset('data/test_sequences.json')
    print_dataset_info(metadata, len(test_entries), dataset_name="Test")

    # Build maze grids for validation
    maze_grids = {}
    for entry in test_entries:
        maze_grids[entry['id']] = load_maze_grid(
            entry['image'], grid_rows=grid_size, grid_cols=grid_size
        )
    print(f"Loaded {len(maze_grids)} maze grids")

    # Load model
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model = ResNetGPT2PrefixModel(
        vocab_size=checkpoint['vocab_size'],
        **cfg['model_kwargs']
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} params")

    # Create dataloader
    test_dataset = MazeDataset(test_entries, tokenizer, transform)
    test_loader = DataLoader(
        test_dataset, batch_size=32, shuffle=False,
        collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id)
    )

    # Run validation
    test_results = evaluate_with_validation(
        model=model, data_loader=test_loader,
        device=device, tokenizer=tokenizer, maze_grids=maze_grids
    )
    print_evaluation_results(test_results, dataset_name=f"{grid_size}x{grid_size} Test")

    # Categorize
    exact_matches = [d for d in test_results['details'] if d['exact_match']]
    creative = [d for d in test_results['details'] if d['valid_solution'] and not d['exact_match']]
    invalid = [d for d in test_results['details'] if not d['valid_solution']]
    creative_pct = 100 * len(creative) / test_results['total']

    # Failure modes
    failure_types = {}
    for d in invalid:
        val = d['validation']
        if val:
            if val.get('hit_wall'): reason = 'hit_wall'
            elif val.get('out_of_bounds'): reason = 'out_of_bounds'
            elif val.get('invalid_token'): reason = 'invalid_token'
            else: reason = 'didnt_reach_goal'
            failure_types[reason] = failure_types.get(reason, 0) + 1

    # Save results
    results_dir = f'results/{grid_size}x{grid_size}'
    os.makedirs(results_dir, exist_ok=True)

    summary = {
        'timestamp': datetime.now().isoformat(),
        'grid_size': f"{grid_size}x{grid_size}",
        'model_config': cfg['model_kwargs'] | {
            'total_parameters': sum(p.numel() for p in model.parameters())
        },
        'performance': {
            'exact_match_pct': test_results['exact_match_pct'],
            'valid_solution_pct': test_results['valid_solution_pct'],
            'creative_solution_pct': creative_pct,
        },
    }

    if 'final_loss' in checkpoint:
        summary['training'] = {
            'final_loss': float(checkpoint['final_loss']),
            'train_accuracy': float(checkpoint.get('train_accuracy', 0)),
            'test_accuracy': float(checkpoint.get('test_accuracy', 0)),
            'generalization_gap': float(checkpoint.get('generalization_gap', 0)),
        }

    with open(f'{results_dir}/summary.json', 'w') as f:
        json.dump(summary, indent=2, fp=f)

    if failure_types:
        with open(f'{results_dir}/failure_analysis.json', 'w') as f:
            json.dump({'total_failures': len(invalid), 'failure_types': failure_types}, indent=2, fp=f)

    print(f"Results saved to {results_dir}/")

    return {
        'grid_size': grid_size,
        'exact_match_pct': test_results['exact_match_pct'],
        'valid_solution_pct': test_results['valid_solution_pct'],
        'creative_pct': creative_pct,
        'total': test_results['total'],
        'loss': checkpoint.get('final_loss', None),
        'train_acc': checkpoint.get('train_accuracy', None),
        'failure_types': failure_types,
    }

## Run Analysis for All Models

In [ ]:
all_results = []

for model_dir in model_dirs:
    grid_size = int(model_dir.split('x')[0])
    checkpoint_path = f'models/{model_dir}/resnet_gpt2_prefix.pth'

    if not os.path.exists(checkpoint_path):
        print(f"Skipping {model_dir} — no checkpoint found")
        continue

    result = analyze_one_model(grid_size, checkpoint_path)
    all_results.append(result)

print(f"\n\nCompleted analysis for {len(all_results)} model(s)")

## Comparison Table

In [ ]:
# Print comparison table
print(f"\n{'='*80}")
print("CROSS-GRID COMPARISON")
print(f"{'='*80}")
print(f"{'Grid':<8} {'Loss':<10} {'Train Acc':<12} {'Exact Match':<14} {'Valid Soln':<14} {'Creative':<12}")
print(f"{'-'*80}")

for r in all_results:
    loss_str = f"{r['loss']:.4f}" if r['loss'] is not None else "N/A"
    train_str = f"{r['train_acc']:.1f}%" if r['train_acc'] is not None else "N/A"
    print(
        f"{r['grid_size']}x{r['grid_size']:<5} "
        f"{loss_str:<10} "
        f"{train_str:<12} "
        f"{r['exact_match_pct']:.1f}%{'':<9} "
        f"{r['valid_solution_pct']:.1f}%{'':<9} "
        f"{r['creative_pct']:.1f}%"
    )

print(f"{'='*80}")
print("\nKey insight: Valid Solution Rate is the true performance metric.")
print("Exact Match underestimates performance because multiple valid paths exist.")

## Comparison Chart

In [ ]:
if len(all_results) > 1:
    labels = [f"{r['grid_size']}x{r['grid_size']}" for r in all_results]
    exact = [r['exact_match_pct'] for r in all_results]
    valid = [r['valid_solution_pct'] for r in all_results]
    creative = [r['creative_pct'] for r in all_results]

    x = np.arange(len(labels))
    width = 0.25

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(x - width, exact, width, label='Exact Match')
    ax.bar(x, valid, width, label='Valid Solution')
    ax.bar(x + width, creative, width, label='Creative Solution')

    ax.set_ylabel('Accuracy (%)')
    ax.set_xlabel('Grid Size')
    ax.set_title('Performance Across Grid Sizes')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('results/comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved to results/comparison.png")
else:
    print("Only one model found — skipping comparison chart.")